# Imports

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Params

In [ ]:
MODEL_DIR = "models"
FRAME_DIR = "models/frames"
LOG_ROOT = "logs"
RUN_ID = ""
MODEL = "drone"
THREADS = 4
TARGET_GSD_M = 0.10
N_WARMUP = 5
N_SHOW = 4
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
UINT8_MAX = 255.0
CLASSES = ["background", "building", "track", "woody", "field", "water"]
PALETTE = [[40, 40, 40], [230, 30, 30], [255, 215, 0], [20, 160, 40], [200, 140, 255], [30, 110, 255]]
MS_PER_S = 1000.0

# Data

In [ ]:
frames = [json.loads(line) for line in (Path(FRAME_DIR) / "frames.jsonl").read_text().splitlines()]
len(frames)

# Model

In [ ]:
def load_session(path: Path, threads: int) -> ort.InferenceSession:
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    options.inter_op_num_threads = 1
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])

In [ ]:
def read_frame(path: Path) -> np.ndarray:
    """Image file → `(H, W, 3)` uint8 RGB."""
    return np.asarray(Image.open(path).convert("RGB"))

In [ ]:
def preprocess(rgb: np.ndarray, mpp: float, input_hw: tuple[int, int]) -> tuple[np.ndarray, tuple[int, int]]:
    """Resample to TARGET_GSD_M, centre-crop / reflect-pad to the model input; returns `(1, 3, H, W)` and the valid size."""
    scale = mpp / TARGET_GSD_M
    size = (round(rgb.shape[1] * scale), round(rgb.shape[0] * scale))
    rgb = np.asarray(Image.fromarray(rgb).resize(size, Image.LANCZOS))
    h, w = input_hw
    top, left = max(0, (rgb.shape[0] - h) // 2), max(0, (rgb.shape[1] - w) // 2)
    rgb = rgb[top:top + h, left:left + w]
    valid_hw = rgb.shape[:2]
    rgb = np.pad(rgb, ((0, h - valid_hw[0]), (0, w - valid_hw[1]), (0, 0)), mode="reflect")
    x = (rgb.astype(np.float32) / UINT8_MAX - np.float32(IMAGENET_MEAN)) / np.float32(IMAGENET_STD)
    return x.transpose(2, 0, 1)[None].copy(), valid_hw

In [ ]:
def postprocess(logits: np.ndarray, valid_hw: tuple[int, int]) -> np.ndarray:
    """`(1, C, H, W)` logits → `(h, w)` uint8 class mask without the padding."""
    return logits[0, :, :valid_hw[0], :valid_hw[1]].argmax(0).astype(np.uint8)

In [ ]:
def run_frame(session: ort.InferenceSession, frame: dict) -> tuple[np.ndarray, dict]:
    """The full onboard flow for one frame, with each stage timed in ms."""
    input_hw = tuple(session.get_inputs()[0].shape[2:])
    t0 = time.perf_counter()
    rgb = read_frame(Path(FRAME_DIR) / frame["file"])
    t1 = time.perf_counter()
    x, valid_hw = preprocess(rgb, frame["mpp"], input_hw)
    t2 = time.perf_counter()
    logits = session.run(None, {"image": x})[0]
    t3 = time.perf_counter()
    mask = postprocess(logits, valid_hw)
    t4 = time.perf_counter()
    stages = {"read_ms": t1 - t0, "preprocess_ms": t2 - t1, "inference_ms": t3 - t2, "postprocess_ms": t4 - t3,
              "total_ms": t4 - t0}
    return mask, {"name": frame["name"], **{k: v * MS_PER_S for k, v in stages.items()}}

In [ ]:
def mask_rgb(mask: np.ndarray) -> np.ndarray:
    return np.uint8(PALETTE)[mask]

# Run

In [ ]:
log_dir = Path(LOG_ROOT) / (RUN_ID or datetime.now().strftime("%Y%m%d-%H%M%S"))
(log_dir / "masks").mkdir(parents=True, exist_ok=True)
log_dir

In [ ]:
start = time.perf_counter()
session = load_session(Path(MODEL_DIR) / f"lraspp-{MODEL}.onnx", THREADS)
model_load_ms = (time.perf_counter() - start) * MS_PER_S
model_load_ms

In [ ]:
for _ in range(N_WARMUP):
    run_frame(session, frames[0])

In [ ]:
masks, timings = {}, []
bar = tqdm(frames, desc="segment")
for frame in bar:
    masks[frame["name"]], row = run_frame(session, frame)
    timings.append(row)
    bar.set_postfix(total_ms=f"{row['total_ms']:.0f}")

In [ ]:
for name, mask in masks.items():
    Image.fromarray(mask_rgb(mask)).save(log_dir / "masks" / f"{name}.png")

In [ ]:
timings = pd.DataFrame(timings).set_index("name")
timings.to_csv(log_dir / "timings.csv")
summary = timings.agg(["median", lambda s: s.quantile(0.9), "max"]).set_axis(["median", "p90", "max"]).round(1)
summary

In [ ]:
(log_dir / "run.json").write_text(json.dumps(
    {"model": MODEL, "threads": THREADS, "onnxruntime": ort.__version__, "n_frames": len(frames),
     "model_load_ms": model_load_ms, "summary": summary.to_dict()}, indent=2))

# Plots

In [ ]:
def plot_frames(names: list[str]) -> plt.Figure:
    fig, axes = plt.subplots(len(names), 2, figsize=(10, 3 * len(names)), squeeze=False)
    for row, name in zip(axes, names):
        frame = next(f for f in frames if f["name"] == name)
        row[0].imshow(read_frame(Path(FRAME_DIR) / frame["file"]))
        row[0].set_title(name)
        row[1].imshow(mask_rgb(masks[name]))
        row[1].set_title(f"mask, {timings.loc[name, 'total_ms']:.0f} ms total")
        for ax in row:
            ax.axis("off")
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_frames(list(masks)[:N_SHOW])
fig.savefig(log_dir / "frames.png")

In [ ]:
ax = timings.drop(columns="total_ms").plot.barh(stacked=True, figsize=(8, 0.35 * len(timings) + 1))
ax.set_xlabel("ms");